In [1]:
from bs4 import BeautifulSoup

In [1]:
import pandas as pd
import numpy as np
from mako.template import Template
# check parent
#for i in df.index:
#    articleID = df.loc[i,'articleID']
#    parentID = df.loc[df.articleID == articleID].parentID.values[0]
#    n = 0
#    while (parentID != 0) & (n < 50):
#        t = articleID
#        articleID = df.loc[df.articleID == articleID].parentID.values[0]
#        parentID = df.loc[df.articleID == articleID].parentID.values[0]
#        n = n+1
#    if parentID != 0:
#        print((parentID,n))

In [ ]:
df0 = pd.read_excel('article1.xlsx', sheet_name=0)
df1 = pd.read_excel('article2.xlsx', sheet_name=0)
df2 = pd.read_excel('article3.xlsx', sheet_name=0)
df3 = pd.read_excel('article4.xlsx', sheet_name=0)
df = pd.concat([df0,df1,df2,df3],ignore_index=True)

df.loc[df.articleID == df.parentID,'parentID'] = 0
df.loc[df.articleID == 1093845093,'parentID'] = 85125490
df.loc[df.articleID == 1244290773,'parentID'] = 92840842
df.loc[df.articleID == 873579830,'parentID'] = 1200400408
df.loc[df.articleID == 890238491,'parentID'] = 888389952
df.loc[df.articleID == 937104000,'parentID'] = 76454218
df.loc[df.articleID == 994402557,'parentID'] = 994311159
df.loc[df.articleID == 994434022,'parentID'] = 994399107

for i in df.index:
    articleID = df.loc[i,'articleID']
    parentID = df.loc[df.articleID == articleID].parentID.values[0]
    n = 0
    while (parentID != 0) & (n < 200):
        t = articleID
        articleID = df.loc[df.articleID == articleID].parentID.values[0]
        parentID = df.loc[df.articleID == articleID].parentID.values[0]
        n = n+1
    if parentID != 0:
        print((parentID,n))
    else:
        df.loc[i,'rootID'] = str(articleID)
# remove false title
df.rootID = df.rootID.astype('int')
df.loc[df.title.isna(),'title'] = ''
df.loc[df.content.isna(),'content'] = ''
df.loc[df.articleID != df.rootID,'content'] = df.loc[df.articleID != df.rootID,'title'] + df.loc[df.articleID != df.rootID,'content']
df.loc[df.articleID != df.rootID,'title'] = ''
for i in df.index:
    parentID = df.loc[i,'parentID']
    df.loc[i,'quote_content'] = ''
    if (parentID != 0) & (parentID != df.loc[i,'rootID']):
        df.loc[i,'quote_content'] = df.loc[df.articleID == parentID].content.values[0]
df.to_pickle('wxmang0209.pkl')

In [2]:
df = pd.read_pickle('wxmang0209.pkl')
        
#df.to_pickle('wxmang.pkl')

In [6]:
id_list = list(df.rootID.drop_duplicates())
for id in id_list:
    genHTML(id,df)

In [5]:
def genHTML(id,df):
    
    df_content = df.loc[df.rootID == id]
    df_article = df_content.loc[df_content.parentID == 0]
    df_comment = df_content.loc[df_content.parentID != 0].sort_values(by='writeTime')

    title = df_article.title.values[0]
    author = df_article.userName.values[0]
    post = df_article.content.values[0].replace('_x000d_','')
    if len(df_article) > 1:
            for i in range(1,len(df_article)):
                title = title + '&' + df_article.title.values[i]
                author = author + '&' + df_article.userName.values[i]
                post = post + '<p><p>-------<p>' + df_article.content.values[i].replace('_x000d_','')

    art_date = (df_article.writeTime.iloc[0]).strftime('%Y-%m-%d %H:%M:%S')
    
    reply_li = []
    for j in df_comment.index:
        content = df_comment.content[j].replace('_x000d_','')
        quote_content = df_comment.quote_content[j]
        comment_date = df_comment.writeTime[j].strftime('%Y-%m-%d %H:%M:%S')
        nickname = df_comment.userName[j]
        comment = content
        if quote_content:
            reply = quote_content.replace('_x000d_','')
        else:
            reply = ''
        reply_li.append((comment,reply,nickname,comment_date))
            
    HTML = Template("""<!DOCTYPE html><html><head>  
    <meta content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=no" name=viewport><meta charset=utf-8>
    <link rel="stylesheet" href="./init.css">
    <link rel="stylesheet" href="./float.css">
    <title>${title}</title>
    </head>
    <body><div class="BODY">
    <div class="BACK"><a href="../index.html">返回索引页</a></div>
    <h1>${title}</h1>
    <p class="AUTHOR">${author}</p>
    <p class="DATE">${date}</p>
    <br><br>
    <div class="POST">${post}</div>
    <div class="REPLY_LI">
    <h2>${len(reply_li)} 条留言</h2>
    %for say, reply, user, time in reply_li:
    <div class="LI">
    <div class="USER"><span class="NAME" >${user}</span><div class="TIME">${time}</div></div>
    <div class="SAY">${say}</div>
    %if reply:
    <div class="REPLY">
    <input type="checkbox" class="exp" id="${time}">
    <div class="text"><label class="btn" for="${time}"></label>
    ${reply}
    </div>
    </div>
    %endif
    </div>
    %endfor
    </div>
    </div>
    <div class="BACK"><a href="../index.html">返回索引页</a></div>
    </body>
    <script src='init.js'></script></html>""")
        
    with open('./html/'+str(id)+'.html','w',encoding='utf8') as f:
        f.write(HTML.render(title=title,date=art_date,post=post,author=author,reply_li=reply_li))
    return

In [199]:
def genINDEX(df):
    df_article = df.loc[df.parentID == 0].sort_values(by='writeTime')
    art_li = []
    for i in df_article.index:
        title = df_article.title[i]
        if title == '':
            continue
        art_id = str(df_article.rootID[i])
        art_date = df_article.writeTime[i].strftime('%Y-%m-%d')
        art_li.append((art_id,title,art_date))

    INDEX = Template("""
    <!DOCTYPE html><html><head>
    <meta content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=no" name=viewport><meta charset=utf-8>
    <link rel="stylesheet" href="./html/init.css">
    <style>
    .LI li{
    list-style-type: disc;
    }
    .LI li{
    margin-bottom:8px;
    }
    .LI a{
    text-decoration: none;
    }
    .LI .title{
    color:#000;
    }
    .LI .title:hover{
    color:#f40;
    }
    .LI .date{
    font-size:12px;
    color:#999;
    float:right;
    }
    </style>
    <title>wxmang的文章</title>
    </head>
    <body><div class="BODY">
    <h1>wxmang的文章</h1>
    <br><br>
    <ul class="LI">
    %for url, name, time in art_li:
    <li><a class="title" href="./html/${url}.html">[${time}]&emsp;${name}</a></li>
    %endfor
    </ul>
    </div></body></html>
    """)

    with open("index.html", "w") as index:
        index.write(INDEX.render(art_li=art_li))
    return

In [200]:
genINDEX(df)

In [60]:
df_article.writeTime.iloc[-1].strftime('%Y')

'2007'

In [13]:
df.loc[(df.content.str.contains('正负电子')) & (df.userName == 'wxmang')].to_excel('search.xlsx')